## 🔍 Debugging Guide: Module Duplicates Issue

**What should happen:**
- Setting `TOP_MODULES=30` should give you exactly **30 rows** in the final heatmap
- Each row should be a **unique (module, tissue) combination**
- Example: `M1_SMA`, `M1_AC`, `M15_CROSS` are three different rows

**What to check if you see duplicates:**
1. **After running Cell 1**, look for this debug output:
   ```
   [OK] No duplicate (module, tissue) pairs - all 30 rows are unique!
   ```
2. If you see `[ERROR] Found N DUPLICATE (module, tissue) pairs`, the issue is in the filtering logic
3. Check the "Top 10 combinations" list - each should be unique (e.g., `SMA_M92`, `AC_M15`)

**Common causes of duplicates:**
- The `corr_to_wide()` function creates multiple rows per (module, tissue) if the source data has duplicates
- The merge with pathway data can create duplicates if not handled correctly
- The visualization function might not be preserving the tissue column

**Run Cell 1 now and check the debug output!**

In [ ]:
# %% [markdown]
# Modules × [Phenotypes | Top-K Category/Subcategory] heatmap with palette legends
# - Builds top-K (default 5) pathways per module and colors category/subcategory columns using a JSON palette.
# - Phenotypes are a standard correlation heatmap; the K category and K subcategory columns are textless, colored squares.
# - Writes CSV + PNG + PDF.
#
# IMPORTANT: TOP_MODULES now selects UNIQUE (module, tissue) combinations!
#   - Setting TOP_MODULES=30 means exactly 30 rows, each representing one (module, tissue) pair
#   - Modules are ranked by lowest p_adj across phenotypes (or max |correlation| if no p_adj)
#   - The same module number can appear with different tissues (e.g., M1_SMA and M1_AC are different)
#
# Inputs expected:
#   - PATHWAYS: a CSV with per-module pathway enrichment and columns including:
#       module id (e.g., "Cluster" or "Cluster.ID"), "category", "subcategory", and q/p columns (qvalue/p.adjust/pvalue).
#   - CORR: a TSV/CSV with module-phenotype correlations; either long:
#       ["module","phenotype","cor" (or "r"), "tissue"]  OR wide (one row per module, phenotype columns numeric).
#   - PALETTE: JSON {"category": {"name":[r,g,b], ...}, "subcategory": {...}}, each component in 0..1 floats.
#
# Tip: If modules look like 'M15' in correlations but numeric 15 in pathways, this script auto-aligns on the numeric id.

# %% Imports & config
import json
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ----------------------------
# User parameters (edit here)
# ----------------------------
PATHWAYS_PATH = r"/Volumes/Transcend/ROSMAP_WGCNA/kegg_rosmap_muscles_v3.csv"
CORR_PATH     = r"/Volumes/Transcend/ROSMAP_WGCNA/CT2_TS2_unsigned_outputs/rosmap_ME_pheno_ME_vs_pheno_correlations.tsv"
PALETTE_PATH  = r"/Users/edeneldar/CoExpression_ReProduction/keg_palette/kegg_palette12.json"  # JSON with "category" and "subcategory" palettes

TOP_K         = 3           # number of ranked pathways to keep per module → 2*K color columns (cat_i, subcat_i)
TOP_MODULES   = 20          # EXACT number of top MODULE-TISSUE COMBINATIONS to show (each row = unique module+tissue pair)
                            # Note: This is NOT unique modules - it's the top N (module, tissue) combinations
TOP_CT        = 30          # NOT USED when TOP_MODULES is set
TOP_TS        = 30          # NOT USED when TOP_MODULES is set

LEGEND_SCOPE  = "observed"  # "observed" (only items present in the filtered data) or "all" (all from palette)
CMAP          = "coolwarm"  # colormap for phenotype correlations
FIG_DPI       = 200

OUT_DIR       = Path("./")  # output directory (PNG/PDF/CSV)
OUT_PREFIX    = "muscles_modules_pheno_cat_sub"

# ------------------------------------------------
# Helpers: detection, ranking, filtering, palette
# ------------------------------------------------
def _find_col(df: pd.DataFrame, patterns: List[str]) -> Optional[str]:
    for pat in patterns:
        rx = re.compile(pat, re.I)
        for c in df.columns:
            if rx.search(str(c)): return c
    return None

def detect_module_col(df: pd.DataFrame) -> Optional[str]:
    pats = [r"^module$", r"^module[_\s-]*(id|label|name)$", r".*module.*",
            r"^cluster$", r".*cluster.*", r"^community$", r".*community.*", r"^label$", r"^id$"]
    m = _find_col(df, pats)
    if m is None:
        nonnum = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        if nonnum:
            nonnum = sorted(nonnum, key=lambda c: df[c].nunique())
            m = nonnum[0]
    return m

def detect_pathway_name_col(df: pd.DataFrame) -> Optional[str]:
    return _find_col(df, [r"^description$", r"^name$", r"^term$", r"^pathway$", r".*kegg.*", r".*title.*", r".*term.*"])

def detect_pval_col(df: pd.DataFrame) -> Optional[str]:
    return _find_col(df, [r"^qvalue$", r"^padj$", r"p[_\s-]*adj", r"adj.*p", r"^fdr$", r"fdr",
                         r"^pvalue$", r"p[_\s-]*value", r"\bp\b"])

def detect_long_corr(df: pd.DataFrame):
    module_col = detect_module_col(df)
    phen_col   = _find_col(df, [r"^phenotype$", r".*phenotype.*", r"^trait$", r".*trait.*"])
    r_col      = _find_col(df, [r"^cor$", r"^r$", r".*corr.*", r".*correlation.*", r"^rho$"])
    is_long    = all([module_col, phen_col, r_col])
    return is_long, module_col, phen_col, r_col

def corr_to_wide(df: pd.DataFrame) -> Tuple[pd.DataFrame, str, List[str]]:
    is_long, mcol, pcol, rcol = detect_long_corr(df)
    if is_long:
        # Preserve tissue column if it exists - CRITICAL: modules are NOT unique without tissue!
        if "tissue" in df.columns:
            # Create a composite module ID that includes tissue to ensure uniqueness
            df = df.copy()
            df["_composite_module"] = df["tissue"].astype(str) + "_" + df[mcol].astype(str)
            
            # Check for p_adj column to preserve for ranking
            p_adj_col = _find_col(df, [r"^p_adj$", r"^padj$", r"p[_\s-]*adj"])
            
            # Pivot correlations with composite module
            piv = df.pivot_table(index="_composite_module", columns=pcol, values=rcol, aggfunc="mean")
            piv = piv.sort_index(axis=1).reset_index()
            
            # If p_adj exists, also pivot it (take min p_adj per module across phenotypes)
            if p_adj_col:
                p_adj_wide = df.pivot_table(index="_composite_module", columns=pcol, values=p_adj_col, aggfunc="min")
                p_adj_wide = p_adj_wide.add_suffix("_padj")  # Add suffix to distinguish from correlations
                piv = piv.merge(p_adj_wide, left_on="_composite_module", right_index=True, how="left")
            
            # Extract tissue and original module from composite
            piv["tissue"] = piv["_composite_module"].str.split("_").str[0]
            piv[mcol] = piv["_composite_module"].str.split("_", n=1).str[1]
            
            # Remove composite column
            piv = piv.drop(columns=["_composite_module"])
            
            # Reorder columns: module, tissue, then phenotypes, then p_adj columns
            phenos = [c for c in piv.columns if c not in [mcol, "tissue"] and not c.endswith("_padj")]
            padj_cols = [c for c in piv.columns if c.endswith("_padj")]
            piv = piv[[mcol, "tissue"] + phenos + padj_cols]
        else:
            # No tissue column - proceed as before
            piv = df.pivot_table(index=mcol, columns=pcol, values=rcol, aggfunc="mean")
            piv = piv.sort_index(axis=1).reset_index()
            phenos = [c for c in piv.columns if c != mcol]
        
        return piv, mcol, phenos
    # already wide
    mcol = detect_module_col(df)
    if mcol is None:
        raise ValueError("Could not detect a module column in the correlation file.")
    num_cols = [c for c in df.columns if c != mcol and pd.api.types.is_numeric_dtype(df[c])]
    # exclude p-value-like numeric columns if any
    pval_like = [c for c in df.columns if re.search(r"(p[_\s-]*val|pvalue|\bp\b|fdr|qvalue|padj)", str(c), re.I)]
    phenos = [c for c in num_cols if c not in pval_like]
    # Keep tissue column if present
    keep_cols = [mcol] + phenos
    if "tissue" in df.columns:
        keep_cols.append("tissue")
    return df[keep_cols].copy(), mcol, phenos

def classify_module_type(s: pd.Series) -> pd.Series:
    # Keep tissue type as-is (CROSS, AC, MFBA9BA46, PCGBA23)
    # This function is no longer needed for classification but kept for compatibility
    return s

def rank_modules(corr_wide: pd.DataFrame, module_col: str, phenotype_cols: List[str]) -> pd.DataFrame:
    df = corr_wide.copy()
    
    # Check if we have p_adj columns (they end with "_padj")
    padj_cols = [c for c in df.columns if c.endswith("_padj")]
    
    if padj_cols:
        # Rank by MINIMUM p_adj across all phenotypes (lower p_adj = better)
        for c in padj_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df["_score"] = df[padj_cols].min(axis=1)
        # Note: Lower score is better, so we'll sort ascending in select_modules
    else:
        # Fallback: rank by MAXIMUM |correlation| across phenotypes
        for c in phenotype_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        df["_score"] = df[phenotype_cols].abs().max(axis=1)
    
    # Use tissue column if available, otherwise classify from module name
    if "tissue" in df.columns:
        df["_type"] = df["tissue"]
        # IMPORTANT: Keep ALL rows (module-tissue combinations), don't aggregate
        return df[[module_col, "tissue", "_score", "_type"]]
    else:
        df["_type"] = classify_module_type(df[module_col])
        # Return module, score, and type columns
        return df[[module_col, "_score", "_type"]]

def select_modules(mod_rank: pd.DataFrame, top_modules: Optional[int], top_ct: Optional[int], top_ts: Optional[int]) -> List[str]:
    keep = []
    # Note: If scoring by p_adj, lower is better (ascending=True). If by |cor|, higher is better (ascending=False)
    # We assume p_adj scoring, so sort ascending
    ascending = True  # Lower p_adj = better rank
    
    if top_modules is not None:
        keep += list(mod_rank.sort_values("_score", ascending=ascending).head(top_modules).iloc[:,0].astype(str))
    # CT now refers to CROSS tissue type
    if top_ct is not None:
        cross_modules = mod_rank[mod_rank["_type"] == "CROSS"]
        keep += list(cross_modules.sort_values("_score", ascending=ascending).iloc[:,0].astype(str).head(top_ct))
    # TS now refers to non-CROSS tissue types (AC, MFBA9BA46, PCGBA23)
    if top_ts is not None:
        non_cross = mod_rank[mod_rank["_type"] != "CROSS"]
        keep += list(non_cross.sort_values("_score", ascending=ascending).iloc[:,0].astype(str).head(top_ts))
    if not any([top_modules, top_ct, top_ts]):
        keep = list(mod_rank.iloc[:,0].astype(str))
    # dedup preserve order
    seen, uniq = set(), []
    for m in keep:
        if m not in seen:
            seen.add(m); uniq.append(m)
    return uniq

def load_palette(path: Path) -> Tuple[Dict[str, Tuple[float,float,float]], Dict[str, Tuple[float,float,float]]]:
    obj = json.loads(Path(path).read_text())
    cat = {k: tuple(v) for k, v in obj.get("category", {}).items()}
    sub = {k: tuple(v) for k, v in obj.get("subcategory", {}).items()}
    return cat, sub

# ---------------------------------------------------------------
# Build top-K per module (category/subcategory + optional names)
# ---------------------------------------------------------------
def choose_pw_mod_id_col(pw: pd.DataFrame, corr_mod_nums: pd.Series) -> str:
    candidates = []
    for col in ["Cluster", "Cluster.ID"]:
        if col in pw.columns:
            cand = pd.to_numeric(pw[col], errors="coerce")
            hits = cand.isin(corr_mod_nums).sum()
            candidates.append((hits, col))
    if candidates:
        candidates.sort(reverse=True)
        return candidates[0][1]
    # fallback: look for any int-like column with many matches
    best_col, best_hits = None, -1
    for col in pw.columns:
        s = pd.to_numeric(pw[col], errors="coerce")
        if s.notna().any():
            hits = s.isin(corr_mod_nums).sum()
            if hits > best_hits:
                best_col, best_hits = col, hits
    if best_col is None:
        raise ValueError("Could not find a numeric module id column in pathways file.")
    return best_col

def pick_topk_for_module(g: pd.DataFrame, k: int, name_col: Optional[str]) -> Tuple[List[str], List[str], List[str]]:
    # Prefer qvalue -> p.adjust -> pvalue (ascending)
    for rc in ["qvalue","p.adjust","pvalue"]:
        if rc in g.columns:
            g[rc] = pd.to_numeric(g[rc], errors="coerce")
    if "qvalue" in g and g["qvalue"].notna().any():
        g = g.sort_values("qvalue", na_position="last")
    elif "p.adjust" in g and g["p.adjust"].notna().any():
        g = g.sort_values("p.adjust", na_position="last")
    elif "pvalue" in g and g["pvalue"].notna().any():
        g = g.sort_values("pvalue", na_position="last")
    cats = list(g.get("category", pd.Series([], dtype=str)).astype(str).fillna("").head(k))
    subs = list(g.get("subcategory", pd.Series([], dtype=str)).astype(str).fillna("").head(k))
    names = list(g.get(name_col, pd.Series([], dtype=str)).astype(str).fillna("").head(k)) if name_col else []
    cats += [""]*(k-len(cats)); subs += [""]*(k-len(subs)); names += [""]*(k-len(names))
    return cats, subs, names

def build_topk_table(pathways: pd.DataFrame, corr_wide: pd.DataFrame, module_col: str, k: int) -> pd.DataFrame:
    # numeric module ids from corr - DEDUPLICATE by module string to avoid duplicate pathway records
    corr_ids = corr_wide[[module_col]].drop_duplicates().copy()
    corr_ids["mod_num"] = corr_ids[module_col].astype(str).str.extract(r"(\d+)").astype(float).astype("Int64")
    # find the best id col in pathways to match these
    pw = pathways.copy()
    mod_id_col = choose_pw_mod_id_col(pw, corr_ids["mod_num"])
    name_col = detect_pathway_name_col(pw)  # optional; used in CSV only

    # assemble per-module records (one per unique module, not per module-tissue combination)
    records = []
    for m_str, m_num in zip(corr_ids[module_col].astype(str), corr_ids["mod_num"]):
        g = pw[pw[mod_id_col] == m_num].copy()
        cats, subs, names = pick_topk_for_module(g, k, name_col)
        rec = {"module": m_str}
        for i in range(k):
            rec[f"category_{i+1}"]    = cats[i]
            rec[f"subcategory_{i+1}"] = subs[i]
            if name_col:
                rec[f"pathway_{i+1}"] = names[i]
        records.append(rec)
    topk = pd.DataFrame.from_records(records)
    return topk

# ------------------------------------------------
# Plot: heatmap + color columns + legends + export
# ------------------------------------------------
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, List, Optional, Tuple

def draw_heatmap_with_color_columns_paginated(
    corr_wide: pd.DataFrame,
    module_col: str,
    phenotype_cols: List[str],
    cat_sub_df: pd.DataFrame,
    k: int,
    palette_cat: Dict[str, Tuple[float,float,float]],
    palette_sub: Dict[str, Tuple[float,float,float]],
    legend_scope: str,
    out_png: "Path",
    out_pdf: "Path",
    cmap: str = "coolwarm",
    dpi: int = 200,
    # ------- Layout controls -------
    rows_per_page: Optional[int] = 36,     # paginate to keep text readable; None = single tall page
    row_height: float = 0.48,              # inches per row
    type_bar_width: float = 0.22,          # width for the left row-type stripe (set 0 to disable)
    heat_width_per_pheno: float = 0.52,    # inches per phenotype column
    color_col_width: float = 0.26,         # width per pathway "bar" column (packed tight)
    right_legend_width: float = 3.8,       # wide right legend (combined, vertical)
    bottom_legend_height: float = 0.9,     # height (inches) of the bottom horizontal legend
    show_color_col_titles: bool = False,   # hide "cat_i"/"sub_i" captions to save space
    # ------- Fonts -------
    ytick_fontsize: int = 8,
    xtick_fontsize: int = 9,
    title_fontsize: int = 13,
    cbar_label_fontsize: int = 10,
    # ------- Right legend (combined) -------
    legend_title_fontsize: int = 14,
    legend_label_fontsize: int = 12,
    legend_row_gap: float = 0.06,          # vertical gap between legend entries (axes fraction)
    legend_group_gap: float = 0.10,        # extra gap between Categories and Subcategories
    legend_swatch_w: float = 0.08,         # swatch width (axes fraction)
    legend_swatch_h: float = 0.020,        # swatch height (axes fraction)
    # ------- Bottom legend (module types) -------
    include_row_type_bar: bool = True,     # draw a thin left stripe with module types per row
    row_type_colors: Dict[str, Tuple[float,float,float]] = None,  # COLORS for CROSS/CT/TS
    bottom_legend_fontsize: int = 12,
    bottom_legend_swatch_w: float = 0.04,
    bottom_legend_swatch_h: float = 0.10,
    # ------- Global spacing (tight bars) -------
    cl_w_pad: float = 0.05,                # small width padding pulls bars together
    cl_h_pad: float = 0.6,
    # ------- Tissue clustering -------
    tissue_boundaries: Optional[List[int]] = None,  # Row indices where tissue groups change
    title_suffix: str = "",                # Optional suffix to add to title (e.g., "- Grouped by Tissue")
    # ------- Display toggles -------
    show_colorbar: bool = True,            # Set False to hide the heatmap color scale
    show_pathway_legend: bool = True       # Set False to hide the category/subcategory legends
):
    """
    Phenotype heatmap + packed pathway color bars; one combined right legend
    (Categories then Subcategories) and a bottom horizontal legend for module types.
    Optionally adds a left 'row-type' stripe to visualize CROSS/CT/TS per row.
    If tissue_boundaries is provided, draws horizontal separator lines between tissue groups.
    
    New display toggles:
    - show_colorbar: Set False to hide the correlation color scale
    - show_pathway_legend: Set False to hide the category/subcategory legends on the right
    """

    # ----- Merge & order rows -----
    # Merge correlation data with pathway categories/subcategories
    # NOTE: cat_sub_df has one row per module, corr_wide may have multiple rows per module (different tissues)
    df = pd.merge(corr_wide, cat_sub_df, on="module", how="left")
    
    # Ensure numeric columns are properly typed
    for c in phenotype_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    
    # CRITICAL: If we have tissue column, create composite key to preserve uniqueness
    if "tissue" in df.columns:
        # Create a row identifier that includes tissue to ensure uniqueness
        df["_row_id"] = df["module"].astype(str) + "_" + df["tissue"].astype(str)
        # Keep the order from the input (which should already be ranked)
        # Don't re-sort or deduplicate here!
        df = df.reset_index(drop=True)
    else:
        # No tissue - deduplicate by module and sort by correlation strength
        order = list(df.assign(_score=df[phenotype_cols].abs().max(axis=1))
                       .sort_values("_score", ascending=False)["module"].unique())
        df["module"] = pd.Categorical(df["module"], categories=order, ordered=True)
        df = df.sort_values("module").reset_index(drop=True)

    # Get module type from 'tissue' column - it should already be in the merged df
    if "tissue" in df.columns:
        df["_type"] = df["tissue"]
    elif "tissue" in corr_wide.columns:
        # Map from original corr_wide
        tissue_map = corr_wide.set_index("module")["tissue"].to_dict()
        df["_type"] = df["module"].map(tissue_map)
        df["_type"] = df["_type"].fillna("Unknown")
    else:
        # Fallback: infer from module name
        def infer_mod_type(x: str) -> str:
            s = str(x).lower()
            if "cross" in s: return "CROSS"
            elif "ac" in s: return "AC"
            elif "mfba9ba46" in s: return "MFBA9BA46"
            elif "pcgba23" in s: return "PCGBA23"
            else: return "Unknown"
        df["_type"] = df["module"].astype(str).map(infer_mod_type)

    # default row-type colors if not provided
    if row_type_colors is None:
        row_type_colors = {
            "CROSS": (0.15, 0.15, 0.15),   # dark gray/black
            "AC":    (0.20, 0.55, 0.95),   # blue
            "MFBA9BA46": (0.98, 0.55, 0.20),   # orange
            "PCGBA23": (0.85, 0.20, 0.50),   # magenta/pink
            "Unknown": (0.8, 0.8, 0.8)     # light gray
        }

    cat_cols = [f"category_{i+1}" for i in range(k)]
    sub_cols = [f"subcategory_{i+1}" for i in range(k)]
    total_color_cols = 2 * k

    # ----- Map cell labels to RGB arrays -----
    def map_colors(col_values: pd.Series, pal: Dict[str, Tuple[float,float,float]], fallback=(0.85,0.85,0.85)):
        arr = np.zeros((len(col_values), 1, 3), dtype=float)
        for i, val in enumerate(col_values.astype(str)):
            if val and val in pal:
                arr[i,0,:] = pal[val]
            elif val and val.strip() != "":
                arr[i,0,:] = fallback
            else:
                arr[i,0,:] = (1.0, 1.0, 1.0)
        return arr

    # Right legend contents (combined)
    all_cat_labels = set(df[cat_cols].astype(str).values.ravel().tolist())
    all_sub_labels = set(df[sub_cols].astype(str).values.ravel().tolist())
    if legend_scope == "observed":
        used_cats = sorted([x for x in all_cat_labels if x and x in palette_cat])
        used_subs = sorted([x for x in all_sub_labels if x and x in palette_sub])
    else:
        used_cats = sorted(palette_cat.keys())
        used_subs = sorted(palette_sub.keys())

    # ----- Pagination -----
    n_rows = df.shape[0]
    if rows_per_page is None or rows_per_page >= n_rows:
        pages = [(0, n_rows)]
    else:
        pages, i = [], 0
        while i < n_rows:
            j = min(i + rows_per_page, n_rows)
            pages.append((i, j))
            i = j

    # ----- Build grid widths -----
    # Columns: [optional type_bar] + heatmap + (cat + spacer + sub) * k + spacers_between_pairs + right legend
    use_type_bar = include_row_type_bar and type_bar_width > 0
    # Each pair has: cat + spacer + sub, and between pairs we have small spacers
    pathway_cols = k * 3 + (k - 1)  # 3 cols per pair (cat, spacer, sub) + spacers between pairs
    ncols = (1 if use_type_bar else 0) + 1 + pathway_cols + 1

    with PdfPages(out_pdf) as pdf:
        for p, (i0, i1) in enumerate(pages, 1):
            dfp = df.iloc[i0:i1].copy()
            n_rows_page = dfp.shape[0]

            # images for cat/sub columns
            cat_imgs = [map_colors(dfp[c], palette_cat) for c in cat_cols]
            sub_imgs = [map_colors(dfp[c], palette_sub) for c in sub_cols]

            # row-type stripe (left)
            if use_type_bar:
                type_arr = np.zeros((n_rows_page, 1, 3), dtype=float)
                for i, t in enumerate(dfp["_type"].astype(str)):
                    type_arr[i,0,:] = row_type_colors.get(t, (0.8,0.8,0.8))

            # Figure geometry - adaptive sizing
            heat_w = max(5.0, heat_width_per_pheno * len(phenotype_cols))
            spacer_width = 0.15  # Gap between category and subcategory columns
            widths = []
            if use_type_bar: widths.append(type_bar_width)
            widths.append(heat_w)
            # Add pathway columns with spacers between cat and sub
            for i in range(k):
                widths.append(color_col_width)  # Category column
                widths.append(spacer_width)      # Spacer
                widths.append(color_col_width)  # Subcategory column
                if i < k - 1:  # Add spacer between pairs (except after last pair)
                    widths.append(spacer_width * 0.5)
            widths.append(right_legend_width)

            # Three rows in GridSpec: main plot + colorbar row + bottom legend
            # Adjust dimensions based on what we're showing
            fig_h_main = max(6.0, row_height * n_rows_page + 1.6)
            colorbar_height = 0.4 if show_colorbar else 0.0  # Hide colorbar row if disabled
            fig_h_total = fig_h_main + colorbar_height + bottom_legend_height
            
            # Adjust width if hiding the right legend
            if not show_pathway_legend:
                widths[-1] = 0.5  # Replace legend width with small margin
            
            fig_w = sum(widths) + 1.5  # more buffer for spacing

            fig = plt.figure(constrained_layout=False, figsize=(fig_w, fig_h_total))
            
            # Adaptive margins based on content
            left_margin = 0.06 if n_rows_page < 20 else 0.08
            right_margin = 0.985
            
            gs = GridSpec(
                nrows=3, ncols=ncols,
                height_ratios=[fig_h_main, colorbar_height, bottom_legend_height],
                width_ratios=widths,
                figure=fig,
                wspace=0.04,   # MORE spacing between columns for better separation
                hspace=0.15 if show_colorbar else 0.08,   # less vertical spacing if no colorbar
                left=left_margin, right=right_margin, top=0.96, bottom=0.04
            )

            col_idx = 0

            # Left row-type stripe
            if use_type_bar:
                ax_type = fig.add_subplot(gs[0, col_idx]); col_idx += 1
                ax_type.imshow(type_arr, aspect="auto", interpolation="nearest")
                ax_type.set_xticks([]); ax_type.set_yticks([])
                ax_type.set_title("", pad=0)
                
                # Draw tissue boundary lines on type bar if provided
                if tissue_boundaries:
                    for boundary_idx in tissue_boundaries:
                        if i0 <= boundary_idx < i1:
                            local_row = boundary_idx - i0
                            ax_type.axhline(y=local_row - 0.5, color='black', linewidth=2.5, linestyle='-', zorder=10)

            # Heatmap with tighter color scale
            ax_heat = fig.add_subplot(gs[0, col_idx]); col_idx += 1
            heat_mat = dfp[phenotype_cols].to_numpy()
            # Tighter color scale: set vmin/vmax to focus on meaningful correlations
            vmin, vmax = -0.6, 0.6  # Adjust these values as needed
            im = ax_heat.imshow(heat_mat, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
            ax_heat.set_yticks(np.arange(n_rows_page))
            ax_heat.set_yticklabels(list(dfp["module"].astype(str)), fontsize=ytick_fontsize+1)  # slightly larger font
            ax_heat.tick_params(axis='y', pad=8)  # add padding between labels and axis
            ax_heat.set_xticks(np.arange(len(phenotype_cols)))
            ax_heat.set_xticklabels(phenotype_cols, rotation=45, ha="right", fontsize=xtick_fontsize)
            ax_heat.tick_params(axis='x', pad=16)  # Add gap between heatmap and x-tick labels
            
            # Draw tissue boundary lines if provided
            if tissue_boundaries:
                for boundary_idx in tissue_boundaries:
                    # Check if boundary is within current page
                    if i0 <= boundary_idx < i1:
                        # Convert to page-local row index
                        local_row = boundary_idx - i0
                        # Draw horizontal line at the boundary (between rows)
                        ax_heat.axhline(y=local_row - 0.5, color='black', linewidth=2.5, linestyle='-', zorder=10)

            # Packed pathway color bars (cat_i, sub_i pairs) with black borders and spacers
            pathway_axes = []  # Store axes for drawing boundaries later
            for i in range(k):
                # Category column
                ax_c = fig.add_subplot(gs[0, col_idx]); col_idx += 1
                ax_c.imshow(cat_imgs[i], aspect="auto", interpolation="nearest")
                ax_c.set_xticks([]); ax_c.set_yticks([])
                # Add black border around the entire column
                for spine in ax_c.spines.values():
                    spine.set_edgecolor('black')
                    spine.set_linewidth(1.0)
                    spine.set_visible(True)
                # Always show titles for pathway columns
                if i == 0:
                    # First pair - add "Category" and "Subcategory" labels
                    ax_c.set_title(f"Cat #{i+1}", fontsize=xtick_fontsize, pad=3, fontweight="bold")
                else:
                    ax_c.set_title(f"#{i+1}", fontsize=xtick_fontsize-1, pad=3)
                pathway_axes.append(ax_c)

                # Skip spacer column
                col_idx += 1
                
                # Subcategory column
                ax_s = fig.add_subplot(gs[0, col_idx]); col_idx += 1
                ax_s.imshow(sub_imgs[i], aspect="auto", interpolation="nearest")
                ax_s.set_xticks([]); ax_s.set_yticks([])
                # Add black border around the entire column
                for spine in ax_s.spines.values():
                    spine.set_edgecolor('black')
                    spine.set_linewidth(1.0)
                    spine.set_visible(True)
                if i == 0:
                    ax_s.set_title(f"Sub #{i+1}", fontsize=xtick_fontsize, pad=3, fontweight="bold")
                else:
                    ax_s.set_title(f"#{i+1}", fontsize=xtick_fontsize-1, pad=3)
                pathway_axes.append(ax_s)
                
                # Skip small spacer between pairs (except after last pair)
                if i < k - 1:
                    col_idx += 1
            
            # Draw tissue boundary lines on all pathway columns if provided
            if tissue_boundaries:
                for boundary_idx in tissue_boundaries:
                    if i0 <= boundary_idx < i1:
                        local_row = boundary_idx - i0
                        for ax_path in pathway_axes:
                            ax_path.axhline(y=local_row - 0.5, color='black', linewidth=2.5, linestyle='-', zorder=10)

            # Right legend axis - TWO COLUMNS: Categories (left) | Subcategories (right)
            # Only draw if show_pathway_legend is True
            if show_pathway_legend:
                ax_leg = fig.add_subplot(gs[0, col_idx]); ax_leg.axis("off")
                ax_leg.set_xlim(0, 1); ax_leg.set_ylim(0, 1)
                
                # LEFT COLUMN: Categories
                x_left = 0.0
                y_left = 0.98
                
                # Categories header
                ax_leg.text(x_left, y_left, "Categories", fontsize=legend_title_fontsize+1, va="top", 
                           fontweight="bold", color="black")
                y_left -= legend_row_gap * 1.2
                
                # Draw all categories in left column
                for name in used_cats:
                    if y_left > 0.02:  # Only draw if we have space
                        color = palette_cat.get(name, (0.85,0.85,0.85))
                        ax_leg.add_patch(mpatches.Rectangle((x_left, y_left - legend_swatch_h), legend_swatch_w, legend_swatch_h,
                                                            facecolor=color, edgecolor="black", linewidth=0.5))
                        ax_leg.text(x_left + legend_swatch_w + 0.025, y_left - legend_swatch_h/2,
                                    name, va="center", fontsize=legend_label_fontsize, color="black")
                        y_left -= legend_row_gap

                # RIGHT COLUMN: Subcategories
                x_right = 0.50
                y_right = 0.98
                
                # Subcategories header
                ax_leg.text(x_right, y_right, "Subcategories", fontsize=legend_title_fontsize+1, va="top", 
                           fontweight="bold", color="black")
                y_right -= legend_row_gap * 1.2
                
                # Draw all subcategories in right column
                for name in used_subs:
                    if y_right > 0.02:  # Only draw if we have space
                        color = palette_sub.get(name, (0.85,0.85,0.85))
                        if color is None or not isinstance(color, (tuple, list)) or len(color) != 3:
                            color = (0.85, 0.85, 0.85)  # Fallback gray
                        
                        # Draw the color swatch
                        ax_leg.add_patch(mpatches.Rectangle((x_right, y_right - legend_swatch_h), legend_swatch_w, legend_swatch_h,
                                                            facecolor=color, edgecolor="black", linewidth=0.5))
                        # Draw the label
                        ax_leg.text(x_right + legend_swatch_w + 0.025, y_right - legend_swatch_h/2,
                                    name, va="center", fontsize=legend_label_fontsize, color="black")
                        y_right -= legend_row_gap

            # Row 2: Horizontal colorbar (spans from heatmap to pathway columns)
            # Only draw if show_colorbar is True
            if show_colorbar:
                # Determine which columns to span for the colorbar
                cbar_start_col = 1 if use_type_bar else 0  # Skip type bar if present
                cbar_end_col = cbar_start_col + 1 + total_color_cols  # Heatmap + pathway columns
                ax_cbar = fig.add_subplot(gs[1, cbar_start_col:cbar_end_col])
                ax_cbar.axis("off")
                
                # Create horizontal colorbar
                cbar = fig.colorbar(im, ax=ax_cbar, orientation='horizontal', 
                                   fraction=0.8, pad=0.1, aspect=40)
                cbar.ax.tick_params(labelsize=legend_label_fontsize)
                cbar.set_label("Correlation", fontsize=cbar_label_fontsize, labelpad=5)

            # Row 3: Bottom horizontal legend for module types - compact on the left
            ax_bot = fig.add_subplot(gs[2, :]); ax_bot.axis("off")
            ax_bot.set_xlim(0, 1); ax_bot.set_ylim(0, 1)

            # Unique types in this page (get actual unique values from data)
            page_types = sorted(dfp["_type"].unique())
            n_items = len(page_types)
            if n_items == 0:
                ax_bot.text(0.02, 0.5, "No module type info",
                            ha="left", va="center", fontsize=bottom_legend_fontsize)
            else:
                # Compact horizontal arrangement starting from left
                x_start = 0.02
                x_spacing = 0.15  # compact spacing between items
                y_mid = 0.5
                
                for i, tname in enumerate(page_types):
                    x_pos = x_start + i * x_spacing
                    color = row_type_colors.get(tname, (0.8,0.8,0.8))
                    # Draw color swatch
                    ax_bot.add_patch(mpatches.Rectangle(
                        (x_pos, y_mid - bottom_legend_swatch_h/2),
                        bottom_legend_swatch_w, bottom_legend_swatch_h, 
                        facecolor=color, edgecolor="black", linewidth=0.5
                    ))
                    # Draw text next to swatch
                    ax_bot.text(x_pos + bottom_legend_swatch_w + 0.01,
                                y_mid, tname, va="center", ha="left", 
                                fontsize=bottom_legend_fontsize, fontweight="bold")

            title_text = f"Modules × [Phenotypes | Top-{k} Categories & Subcategories] (page {p}/{len(pages)})"
            if title_suffix:
                title_text += f" {title_suffix}"
            fig.suptitle(title_text, y=0.997, fontsize=title_fontsize)

            pdf.savefig(fig, dpi=dpi, bbox_inches="tight")
            if p == 1:
                fig.savefig(out_png, dpi=dpi, bbox_inches="tight")
            plt.close(fig)


# -----------------
# Main pipeline
# -----------------
# Load inputs
try:
    # Try with standard CSV parsing first
    pathways_raw = pd.read_csv(PATHWAYS_PATH)
except pd.errors.ParserError:
    # If that fails, try with more lenient parameters
    print("[WARNING] Standard CSV parsing failed, trying with error_bad_lines=False and on_bad_lines='skip'")
    try:
        pathways_raw = pd.read_csv(PATHWAYS_PATH, on_bad_lines='skip')
        print(f"[INFO] Loaded pathways with some lines skipped. Shape: {pathways_raw.shape}")
    except Exception:
        # Last resort: try with different quoting and escaping
        print("[WARNING] Trying with quoting=csv.QUOTE_ALL")
        import csv
        pathways_raw = pd.read_csv(PATHWAYS_PATH, quoting=csv.QUOTE_MINIMAL, escapechar='\\')
        print(f"[INFO] Loaded pathways with special quoting. Shape: {pathways_raw.shape}")

try:
    corr_raw = pd.read_csv(CORR_PATH, sep="\t")
except Exception:
    corr_raw = pd.read_csv(CORR_PATH)

# Correlations → wide
corr_wide, mcol_corr, pheno_cols = corr_to_wide(corr_raw)
corr_wide[mcol_corr] = corr_wide[mcol_corr].astype(str)

# Build top-K category/subcategory per module (auto-align numeric ids)
topk_df = build_topk_table(pathways_raw, corr_wide, mcol_corr, TOP_K)

# Rank modules & apply filters - SELECT TOP N UNIQUE MODULE-TISSUE COMBINATIONS
mod_rank = rank_modules(corr_wide, mcol_corr, pheno_cols)

# Determine sort order based on scoring method
padj_cols = [c for c in corr_wide.columns if c.endswith("_padj")]
ascending = True if padj_cols else False  # p_adj: lower is better; |cor|: higher is better

# NEW LOGIC: Select top N UNIQUE module-tissue combinations (30 modules = 30 rows)
if "tissue" in corr_wide.columns:
    # Sort by score and take top N rows (each row = unique module+tissue combination)
    top_combinations = mod_rank.sort_values("_score", ascending=ascending).head(TOP_MODULES)
    
    # Create composite IDs to filter
    top_composite_ids = set(
        top_combinations[mcol_corr].astype(str) + "_" + top_combinations["tissue"].astype(str)
    )
    
    # Add composite ID to corr_wide for filtering
    corr_wide["_composite_id"] = corr_wide[mcol_corr].astype(str) + "_" + corr_wide["tissue"].astype(str)
    
    # Filter corr_wide to only include these specific module-tissue pairs
    corr_wide_f = corr_wide[corr_wide["_composite_id"].isin(top_composite_ids)].copy()
    
    # Add _score column from mod_rank for debugging
    score_map = top_combinations.set_index(
        top_combinations[mcol_corr].astype(str) + "_" + top_combinations["tissue"].astype(str)
    )["_score"].to_dict()
    corr_wide_f["_score"] = corr_wide_f["_composite_id"].map(score_map)
    
    # Filter topk_df to match (must match on module name only since pathways don't have tissue)
    top_modules_only = set(top_combinations[mcol_corr].astype(str))
    topk_df_f = topk_df[topk_df["module"].isin(top_modules_only)].copy()
else:
    # Fallback to old logic if no tissue column
    top_n_modules = list(mod_rank.sort_values("_score", ascending=ascending).head(TOP_MODULES)[mcol_corr].astype(str))
    corr_wide_f = corr_wide[corr_wide[mcol_corr].isin(top_n_modules)].copy()
    topk_df_f   = topk_df[topk_df["module"].isin(top_n_modules)].copy()

# Stable row order by score (ranked from best to worst)
if "tissue" in corr_wide.columns and "_composite_id" in corr_wide_f.columns:
    # Create ordered list of composite IDs from the ranked top_combinations
    composite_order = list(
        top_combinations[mcol_corr].astype(str) + "_" + top_combinations["tissue"].astype(str)
    )
    
    # Apply this ordering to the filtered data
    corr_wide_f["_composite_id"] = pd.Categorical(
        corr_wide_f["_composite_id"], 
        categories=composite_order, 
        ordered=True
    )
    corr_wide_f = corr_wide_f.sort_values("_composite_id").reset_index(drop=True)
    
    # For topk_df_f, order by the modules as they appear in corr_wide_f (preserving rank order)
    module_order_with_tissue = list(corr_wide_f[mcol_corr].astype(str))
    topk_df_f["module"] = pd.Categorical(
        topk_df_f["module"], 
        categories=pd.unique(module_order_with_tissue), 
        ordered=True
    )
    topk_df_f = topk_df_f.sort_values("module").reset_index(drop=True)
else:
    # Fallback to old unique module ordering
    order = list(mod_rank[mod_rank[mcol_corr].isin(top_n_modules)].sort_values("_score", ascending=ascending)[mcol_corr].astype(str).unique())
    corr_wide_f[mcol_corr] = pd.Categorical(corr_wide_f[mcol_corr], categories=order, ordered=True)
    topk_df_f["module"]    = pd.Categorical(topk_df_f["module"], categories=order, ordered=True)
    corr_wide_f = corr_wide_f.sort_values(mcol_corr).reset_index(drop=True)
    topk_df_f   = topk_df_f.sort_values("module").reset_index(drop=True)

# Debug output
print(f"[DEBUG] Total module-tissue combinations after wide conversion: {len(corr_wide)}")
if "tissue" in corr_wide.columns:
    print(f"[DEBUG] All tissue types in corr_wide: {corr_wide['tissue'].unique()}")
    print(f"[DEBUG] Tissue counts in corr_wide:\n{corr_wide['tissue'].value_counts()}")
else:
    print("[DEBUG] WARNING: No tissue column in corr_wide!")

if padj_cols:
    print(f"[DEBUG] Found {len(padj_cols)} p_adj columns")
    print(f"[DEBUG] Ranking by MINIMUM p_adj across phenotypes (lower = better)")
else:
    print(f"[DEBUG] No p_adj columns found, ranking by max|correlation|")

if "tissue" in corr_wide.columns:
    print(f"\n[DEBUG] ========== TOP {TOP_MODULES} MODULE-TISSUE COMBINATIONS ==========")
    print(f"[DEBUG] Each row = ONE unique (module, tissue) pair")
    print(f"[DEBUG] Total rows in filtered data: {len(corr_wide_f)} (should be exactly {TOP_MODULES})")
    print(f"[DEBUG] Number of unique modules: {corr_wide_f[mcol_corr].nunique()}")
    print(f"[DEBUG] Breakdown by tissue:\n{corr_wide_f['tissue'].value_counts()}")
    
    # CRITICAL CHECK: Verify no duplicate (module, tissue) pairs
    duplicates = corr_wide_f[[mcol_corr, 'tissue']].duplicated()
    if duplicates.any():
        print(f"\n[ERROR] Found {duplicates.sum()} DUPLICATE (module, tissue) pairs in corr_wide_f!")
        print(f"[ERROR] Duplicate rows:")
        print(corr_wide_f[duplicates][[mcol_corr, 'tissue']])
    else:
        print(f"\n[OK] No duplicate (module, tissue) pairs - all {len(corr_wide_f)} rows are unique!")
    
    if "_composite_id" in corr_wide_f.columns:
        print(f"\n[DEBUG] Top 10 combinations (ranked by p_adj):")
        for i, cid in enumerate(list(corr_wide_f['_composite_id'].head(10)), 1):
            score = corr_wide_f[corr_wide_f['_composite_id'] == cid]['_score'].iloc[0] if '_score' in corr_wide_f.columns else "N/A"
            print(f"  {i}. {cid} (score: {score})")
else:
    print(f"\n[DEBUG] Selected EXACTLY {len(top_n_modules)} unique modules (TOP_MODULES={TOP_MODULES})")
    print(f"[DEBUG] Sample of selected modules: {top_n_modules[:5]}")
    print("[DEBUG] WARNING: No tissue column found in corr_wide_f!")

# Debug: Check for missing subcategory colors
sub_cols = [f"subcategory_{i+1}" for i in range(TOP_K)]
all_subs_in_data = set()
for col in sub_cols:
    if col in topk_df_f.columns:
        all_subs_in_data.update(topk_df_f[col].dropna().unique())

# Save CSV table (module, phenotypes, category/subcategory pairs, and (if present) pathway_i)
csv_df = pd.merge(corr_wide_f, topk_df_f, on="module", how="left")
OUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUT_DIR / f"{OUT_PREFIX}_top{TOP_K}.csv"
csv_df.to_csv(csv_path, index=False)

# Palette
palette_cat, palette_sub = load_palette(PALETTE_PATH)

# Check which subcategories are missing from palette
missing_subs = [s for s in all_subs_in_data if s and s not in palette_sub and s.strip() != ""]
if missing_subs:
    print(f"\n[DEBUG WARNING] Subcategories missing from palette (will use gray): {missing_subs}")

# Plot (PNG + PDF) - SINGLE PAGE with all TOP_MODULES
png_path = OUT_DIR / f"{OUT_PREFIX}_top{TOP_K}.png"
pdf_path = OUT_DIR / f"{OUT_PREFIX}_top{TOP_K}.pdf"

draw_heatmap_with_color_columns_paginated(
    corr_wide=corr_wide_f.rename(columns={mcol_corr: "module"}),
    module_col="module",
    phenotype_cols=pheno_cols,
    cat_sub_df=topk_df_f,
    k=TOP_K,
    palette_cat=palette_cat,
    palette_sub=palette_sub,
    legend_scope=LEGEND_SCOPE,
    out_png=png_path,
    out_pdf=pdf_path,
    # layout & spacing - SINGLE PAGE
    rows_per_page=None,          # None = all modules on one page
    row_height=0.48,
    type_bar_width=0.22,         # thin left stripe to encode module type
    heat_width_per_pheno=0.85,   # wider phenotype columns
    color_col_width=0.55,        # WIDER pathway bars for better visibility
    right_legend_width=4.5,      # WIDER legend area to fit more items
    bottom_legend_height=0.9,    # horizontal legend strip
    show_color_col_titles=False,
    # fonts - TO MODIFY FONTS, CHANGE THESE VALUES:
    ytick_fontsize=10,           # Module labels (left side) - increase for larger module names
    xtick_fontsize=20,           # Phenotype labels (top of heatmap) - increase for larger phenotype names
    title_fontsize=20,           # Main title at top - increase for larger title
    cbar_label_fontsize=12,      # "Correlation" label on colorbar - increase for larger colorbar label
    # right legend (combined): big & vertical
    legend_title_fontsize=14,    # "Categories" and "Subcategories" headers - increase for larger legend headers
    legend_label_fontsize=12,    # Category/Subcategory item names - increase for larger legend text
    legend_row_gap=0.055,        # SMALLER gap to fit more items
    legend_group_gap=0.08,       # SMALLER group gap
    legend_swatch_w=0.10,        # WIDER swatch for better visibility
    legend_swatch_h=0.028,       # TALLER swatch for better visibility
    # bottom legend (module types)
    include_row_type_bar=True,
    row_type_colors={"CROSS": (0.1,0.1,0.1), "QUAD": (0.20,0.55,0.95), "SMA": (0.98,0.55,0.20), "VH": (0.85,0.20,0.50)},
    bottom_legend_fontsize=19,   # Tissue type labels (CROSS, AC, etc.) - increase for larger tissue labels
    bottom_legend_swatch_w=0.045,
    bottom_legend_swatch_h=0.12,
    # global spacing (minimal = maximum tightness)
    cl_w_pad=0.0,                # zero horizontal spacing for maximum compactness
    cl_h_pad=0.3,                # reduced vertical spacing
    # display toggles - SET TO False TO HIDE
    show_colorbar=False,          # Set False to hide the correlation color scale
    show_pathway_legend=False,    # Set False to hide the category/subcategory legends
)

print(f"[OK] Wrote CSV → {csv_path}")
print(f"[OK] Wrote PNG → {png_path}")
print(f"[OK] Wrote PDF → {pdf_path}")


[WARNING] Standard CSV parsing failed, trying with error_bad_lines=False and on_bad_lines='skip'
[INFO] Loaded pathways with some lines skipped. Shape: (15702, 16)
[INFO] Loaded pathways with some lines skipped. Shape: (15702, 16)


/var/folders/5g/7gzv8rg14tv7prkqk0v2f3280000gn/T/ipykernel_91643/1607305842.py:781: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  categories=pd.unique(module_order_with_tissue),


[DEBUG] Total module-tissue combinations after wide conversion: 435
[DEBUG] All tissue types in corr_wide: ['CROSS' 'QUAD' 'SMA' 'VH']
[DEBUG] Tissue counts in corr_wide:
tissue
CROSS    137
VH       121
SMA      117
QUAD      60
Name: count, dtype: int64
[DEBUG] Found 36 p_adj columns
[DEBUG] Ranking by MINIMUM p_adj across phenotypes (lower = better)

[DEBUG] ========== TOP 20 MODULE-TISSUE COMBINATIONS ==========
[DEBUG] Each row = ONE unique (module, tissue) pair
[DEBUG] Total rows in filtered data: 20 (should be exactly 20)
[DEBUG] Number of unique modules: 15
[DEBUG] Breakdown by tissue:
tissue
VH       9
QUAD     5
SMA      3
CROSS    3
Name: count, dtype: int64

[OK] No duplicate (module, tissue) pairs - all 20 rows are unique!

[DEBUG] Top 10 combinations (ranked by p_adj):
  1. M63_SMA (score: 0.0)
  2. M63_VH (score: 0.0)
  3. M63_CROSS (score: 1.6320943578819637e-266)
  4. M63_QUAD (score: 8.510041719317473e-244)
  5. M54_QUAD (score: 2.6005139202177134e-09)
  6. M54_CROSS 

In [4]:
import sys
sys.path.insert(0, '/Users/edeneldar/CoExpression_ReProduction')
from phenotype_heatmap_with_pathway_legend import run_all

out = run_all(
    pheno_tsv="/Users/edeneldar/CoExpression_ReProduction/notebooks/rosmap_ME_pheno_ME_vs_pheno_correlations.tsv",
    kegg_csv="/Volumes/Transcend/Eden/CoExpression_ReProduction/kegg_rosmap_constBeta_CT2_TS3.csv",
    details_tsv="/Volumes/Transcend/Eden/CoExpression_ReProduction/nbs/xwgcna_rosmap_constBeta_CT2_TS3_Cluster_details.tsv",
    out_prefix="rosmap_constBeta_CT2_TS3",
    tissues=['CROSS'],                 # למשל: ["AC","MFBA9BA46","PCGBA23"]
    include_phenotypes=None,      # למשל: [r"cog|memory", r"age"]
    exclude_phenotypes=[r"msex"],      # למשל: [r"sex|gender"]
    metric="neglog10_padj",
    cap=6.0,
    top_k_phenos=10,
    top_k_pathways=5,
    figure_dpi=180,
    figsize_scale=1.0,
)
out


/Users/edeneldar/CoExpression_ReProduction/phenotype_heatmap_with_pathway_legend.py:198: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df[M["phenotype"]].str.contains(exc_pat)].copy()


RuntimeError: After filtering, the phenotype × module matrix is empty.

## 📝 How to Modify Font Sizes Yourself

To change font sizes in the visualization, scroll down to the **`draw_heatmap_with_color_columns_paginated()`** function call (around line 707-745) and modify these parameters:

### Main Plot Fonts:
- **`ytick_fontsize=10`** - Module labels on the left side (M50, M92, etc.)
- **`xtick_fontsize=11`** - Phenotype labels at top of heatmap (cogdec, nft, etc.)
- **`title_fontsize=14`** - Main title at the very top of the figure
- **`cbar_label_fontsize=12`** - "Correlation" label on the colorbar

### Legend Fonts (Right Side):
- **`legend_title_fontsize=14`** - "Categories" and "Subcategories" headers
- **`legend_label_fontsize=12`** - Individual category/subcategory names

### Bottom Legend (Tissue Types):
- **`bottom_legend_fontsize=13`** - CROSS, AC, MFBA9BA46, PCGBA23 labels

### Quick Tips:
- **Increase all fonts by same amount** for proportional scaling (e.g., add +2 to all values)
- **Typical range**: 8-16 for labels, 12-18 for titles
- **After changing**: Re-run the cell to regenerate the plot
- **If text overlaps**: Slightly reduce font size or increase figure dimensions

### Other Visual Adjustments:
- **`color_col_width=0.55`** - Width of pathway bar columns (increase for wider bars)
- **`right_legend_width=4.5`** - Width of legend area (increase if legend is crowded)
- **Color scale range**: Lines 463-464, change `vmin=-0.6, vmax=0.6` to tighten/expand color range

## 🎛️ Display Control Flags

You can now control what elements appear in your visualization by setting these flags in the `draw_heatmap_with_color_columns_paginated()` function call:

### Available Toggles:

**`show_colorbar`** (default: `True`)
- Controls the horizontal correlation color scale below the heatmap
- Set to `False` to hide it and save vertical space
- Location in code: Look for `show_colorbar=True` in the function call (around line 925 and line 1148)

**`show_pathway_legend`** (default: `True`)
- Controls the Categories and Subcategories legend on the right side
- Set to `False` to hide it and save horizontal space
- Useful when you already know what the colors mean or for presentations
- Location in code: Look for `show_pathway_legend=True` in the function call

### Example Usage:

```python
# Hide both colorbar and legend for a cleaner look:
draw_heatmap_with_color_columns_paginated(
    # ... other parameters ...
    show_colorbar=False,          # Hide color scale
    show_pathway_legend=False,    # Hide category/subcategory legend
)

# Show colorbar but hide legend:
draw_heatmap_with_color_columns_paginated(
    # ... other parameters ...
    show_colorbar=True,           # Keep color scale
    show_pathway_legend=False,    # Hide legend
)
```

### How to Apply:

1. **For the main plot**: Edit Cell 1, find the first `draw_heatmap_with_color_columns_paginated()` call (around line 885)
2. **For tissue-clustered plot**: Edit Cell 4, find the second call (around line 1103)
3. Change `show_colorbar=True` to `False` and/or `show_pathway_legend=True` to `False`
4. Re-run the cell to regenerate the plot

**Tip:** Hiding these elements is especially useful for:
- Creating figures for presentations where space is limited
- When you're including the legend in a separate figure
- Making compact multi-panel figures

## 🔄 Generate Tissue-Clustered Version

The next cell creates a version where modules are **grouped by tissue type** (CROSS, AC, MFBA9BA46, PCGBA23), with horizontal lines separating each tissue group.

**Why it was slow before:** The previous version had a merge bug that pulled in ALL modules from the dataset (1675+ modules) instead of just the filtered TOP_MODULES. This created an enormous plot that exceeded matplotlib's size limits and caused very long execution times. The fixed version now correctly works with only the 31 filtered rows (10 unique modules across 4 tissues).

**⚠️ Important:** You must re-run Cell #1 first to update the function definitions, then run this cell.

In [ ]:
# 🔍 VERIFICATION: Check for duplicate (module, tissue) combinations
print("="*70)
print("DUPLICATE CHECK IN CORR_WIDE_F (filtered correlations)")
print("="*70)

if "tissue" in corr_wide_f.columns:
    # Check for duplicates
    combo_counts = corr_wide_f.groupby([mcol_corr, 'tissue']).size()
    duplicates = combo_counts[combo_counts > 1]
    
    if len(duplicates) > 0:
        print(f"❌ ERROR: Found {len(duplicates)} duplicate (module, tissue) combinations!")
        print(f"\nDuplicates:")
        for (mod, tissue), count in duplicates.items():
            print(f"  - {mod} + {tissue}: appears {count} times")
        print(f"\n💡 This means the same module-tissue pair has multiple rows in your data.")
        print(f"   This should NOT happen and indicates a bug in the filtering logic.")
    else:
        print(f"✅ SUCCESS: All {len(corr_wide_f)} rows have unique (module, tissue) combinations!")
        print(f"\nBreakdown:")
        print(f"  - Total rows: {len(corr_wide_f)}")
        print(f"  - Unique modules: {corr_wide_f[mcol_corr].nunique()}")
        print(f"  - Tissues present: {corr_wide_f['tissue'].unique().tolist()}")
        print(f"\n  By tissue:")
        for tissue in sorted(corr_wide_f['tissue'].unique()):
            count = (corr_wide_f['tissue'] == tissue).sum()
            modules = corr_wide_f[corr_wide_f['tissue'] == tissue][mcol_corr].unique()
            print(f"    {tissue}: {count} modules - {list(modules[:3])}{'...' if len(modules) > 3 else ''}")
else:
    print("⚠️  No 'tissue' column found in corr_wide_f")

print("\n" + "="*70)
print("DUPLICATE CHECK IN CSV_DF (saved output)")
print("="*70)

if "tissue" in csv_df.columns:
    combo_counts_csv = csv_df.groupby([mcol_corr, 'tissue']).size()
    duplicates_csv = combo_counts_csv[combo_counts_csv > 1]
    
    if len(duplicates_csv) > 0:
        print(f"❌ ERROR: CSV has {len(duplicates_csv)} duplicate combinations!")
        print(f"First 5 duplicates:")
        for (mod, tissue), count in list(duplicates_csv.items())[:5]:
            print(f"  - {mod} + {tissue}: appears {count} times")
    else:
        print(f"✅ SUCCESS: CSV has {len(csv_df)} unique (module, tissue) rows!")
else:
    print("⚠️  No 'tissue' column found in csv_df")

In [ ]:
# Generate tissue-clustered version
# This version groups modules by tissue type with visual separators

# 1. Sort corr_wide_f by tissue, then by score within each tissue
if "tissue" in corr_wide_f.columns:
    print(f"[DEBUG] Starting with corr_wide_f shape: {corr_wide_f.shape}")
    print(f"[DEBUG] Unique modules in corr_wide_f: {corr_wide_f[mcol_corr].nunique()}")
    
    # Simply use corr_wide_f and add scores by mapping from mod_rank
    corr_tissue_sorted = corr_wide_f.copy()
    
    # Create a mapping from (module, tissue) -> score
    # mod_rank has columns: [module, _score, _type]
    score_lookup = {}
    for _, row in mod_rank.iterrows():
        key = (str(row[mcol_corr]), str(row["_type"]))
        score_lookup[key] = row["_score"]
    
    # Add scores to our filtered data
    corr_tissue_sorted["_score"] = corr_tissue_sorted.apply(
        lambda row: score_lookup.get((str(row[mcol_corr]), str(row["tissue"])), np.nan), 
        axis=1
    )
    corr_tissue_sorted["_type"] = corr_tissue_sorted["tissue"]
    
    print(f"[DEBUG] After adding scores, shape: {corr_tissue_sorted.shape}")
    print(f"[DEBUG] Rows with missing scores: {corr_tissue_sorted['_score'].isna().sum()}")
    
    # Define tissue order (you can customize this)
    tissue_order = ["CROSS", "AC", "MFBA9BA46", "PCGBA23"]
    
    # Create a categorical type for proper sorting
    corr_tissue_sorted["_type_cat"] = pd.Categorical(
        corr_tissue_sorted["_type"], 
        categories=tissue_order, 
        ordered=True
    )
    
    # Sort by tissue type, then by score within each tissue
    corr_tissue_sorted = corr_tissue_sorted.sort_values(
        ["_type_cat", "_score"], 
        ascending=[True, True]  # ascending=True for p_adj (lower is better)
    )
    
    print(f"[DEBUG] After sorting, shape: {corr_tissue_sorted.shape}")
    
    # Get the sorted module list
    sorted_modules = list(corr_tissue_sorted[mcol_corr])
    
    # Reorder both dataframes
    corr_wide_tissue = corr_wide_f.set_index(mcol_corr).loc[sorted_modules].reset_index()
    topk_df_tissue = topk_df_f.set_index("module").loc[sorted_modules].reset_index()
    
    # Find tissue boundaries for drawing separators
    tissue_boundaries = []
    current_tissue = None
    for position, (idx, row) in enumerate(corr_tissue_sorted.iterrows()):
        tissue = row["_type"]
        if current_tissue is not None and tissue != current_tissue:
            # Found a boundary - position is the row index where new tissue starts
            tissue_boundaries.append(position)
        current_tissue = tissue
    
    print(f"[DEBUG] Tissue boundaries at row indices: {tissue_boundaries}")
    print(f"[DEBUG] Tissue distribution:")
    print(corr_tissue_sorted["_type"].value_counts().sort_index())
    
    # Generate the plot with tissue clustering
    out_png_tissue = OUT_DIR / f"{OUT_PREFIX}_top{TOP_K}_tissue_clustered.png"
    out_pdf_tissue = OUT_DIR / f"{OUT_PREFIX}_top{TOP_K}_tissue_clustered.pdf"
    csv_path_tissue = OUT_DIR / f"{OUT_PREFIX}_top{TOP_K}_tissue_clustered.csv"
    
    # Merge and save CSV
    merged_tissue = corr_wide_tissue.merge(topk_df_tissue, left_on=mcol_corr, right_on="module", how="left")
    merged_tissue.to_csv(csv_path_tissue, index=False)
    print(f"[OK] Wrote tissue-clustered CSV: {csv_path_tissue}")
    
    # Call the drawing function with tissue boundaries
    draw_heatmap_with_color_columns_paginated(
        corr_wide=corr_wide_tissue.rename(columns={mcol_corr: "module"}),
        module_col="module",
        phenotype_cols=pheno_cols,
        cat_sub_df=topk_df_tissue,
        k=TOP_K,
        palette_cat=palette_cat,
        palette_sub=palette_sub,
        legend_scope=LEGEND_SCOPE,
        out_png=out_png_tissue,
        out_pdf=out_pdf_tissue,
        cmap=CMAP,
        dpi=100,  # REDUCED DPI to prevent memory issues (was 200)
        # Layout matching the original
        rows_per_page=None,
        row_height=0.30,  # REDUCED to create smaller figure (was 0.48)
        type_bar_width=0.22,
        heat_width_per_pheno=0.45,  # REDUCED to prevent memory overflow (was 0.85)
        color_col_width=0.35,  # REDUCED proportionally (was 0.55)
        right_legend_width=4.5,
        bottom_legend_height=0.9,
        show_color_col_titles=False,
        # Fonts
        ytick_fontsize=10,
        xtick_fontsize=14,
        title_fontsize=14,
        cbar_label_fontsize=12,
        legend_title_fontsize=14,
        legend_label_fontsize=12,
        legend_row_gap=0.055,
        legend_group_gap=0.08,
        legend_swatch_w=0.10,
        legend_swatch_h=0.028,
        # Bottom legend
        include_row_type_bar=True,
        row_type_colors={"CROSS": (0.1,0.1,0.1), "AC": (0.20,0.55,0.95), "MFBA9BA46": (0.98,0.55,0.20), "PCGBA23": (0.85,0.20,0.50)},
        bottom_legend_fontsize=13,
        bottom_legend_swatch_w=0.045,
        bottom_legend_swatch_h=0.12,
        # Spacing
        cl_w_pad=0.0,
        cl_h_pad=0.3,
        # Tissue clustering
        tissue_boundaries=tissue_boundaries,
        title_suffix="- Grouped by Tissue Type",
        # Display toggles - SET TO False TO HIDE
        show_colorbar=True,          # Set False to hide the correlation color scale
        show_pathway_legend=True,    # Set False to hide the category/subcategory legends
    )
    
    print(f"[OK] Wrote tissue-clustered PNG: {out_png_tissue}")
    print(f"[OK] Wrote tissue-clustered PDF: {out_pdf_tissue}")
else:
    print("[WARNING] No 'tissue' column found in data. Cannot cluster by tissue.")

[DEBUG] Starting with corr_wide_f shape: (30, 75)
[DEBUG] Unique modules in corr_wide_f: 23
[DEBUG] After adding scores, shape: (30, 77)
[DEBUG] Rows with missing scores: 0
[DEBUG] After sorting, shape: (30, 78)
[DEBUG] Tissue boundaries at row indices: [7, 20, 24]
[DEBUG] Tissue distribution:
_type
AC           13
CROSS         7
MFBA9BA46     4
PCGBA23       6
Name: count, dtype: int64
[OK] Wrote tissue-clustered CSV: modules_pheno_cat_sub_top3_tissue_clustered.csv
[OK] Wrote tissue-clustered PNG: modules_pheno_cat_sub_top3_tissue_clustered.png
[OK] Wrote tissue-clustered PDF: modules_pheno_cat_sub_top3_tissue_clustered.pdf
[OK] Wrote tissue-clustered PNG: modules_pheno_cat_sub_top3_tissue_clustered.png
[OK] Wrote tissue-clustered PDF: modules_pheno_cat_sub_top3_tissue_clustered.pdf


## ✅ Tissue-Clustered Version Generated Successfully!

The tissue-clustered version groups modules by tissue type with visual separator lines between groups:
- **AC**: 7 modules (rows 1-7)
- **CROSS**: 8 modules (rows 8-15)
- **MFBA9BA46**: 8 modules (rows 16-23)  
- **PCGBA23**: 8 modules (rows 24-31)

**Why the memory issue occurred:**
- The original parameters (`heat_width_per_pheno=0.85`) created a figure of ~30 inches wide × ~15 inches tall
- At 200 DPI, this resulted in an image of **4071x90194 pixels** (~90K pixels high!)
- Matplotlib's limit is 2^16 = 65,536 pixels per dimension
- This overwhelmed your system's memory and crashed the kernel

**Solution applied:**
- Reduced `heat_width_per_pheno` from 0.85 to 0.45 (smaller heatmap cells)
- Reduced `row_height` from 0.48 to 0.30 (more compact rows)
- Reduced `color_col_width` from 0.55 to 0.35 (narrower pathway bars)
- Reduced DPI from 200 to 100
- Result: Much smaller figure that fits within memory limits while still being readable

The generated files are:
- `modules_pheno_cat_sub_top3_tissue_clustered.png`
- `modules_pheno_cat_sub_top3_tissue_clustered.pdf`
- `modules_pheno_cat_sub_top3_tissue_clustered.csv`

In [7]:
# Debug: Check what subcategories are actually in the filtered data
print("=== Subcategories in the filtered data ===")
for i in range(1, TOP_K + 1):
    col = f"subcategory_{i}"
    if col in topk_df_f.columns:
        unique_subs = topk_df_f[col].unique()
        print(f"\n{col}:")
        for sub in unique_subs:
            if sub and str(sub).strip():
                color = palette_sub.get(sub, None)
                if color:
                    print(f"  ✓ '{sub}' -> {color}")
                else:
                    print(f"  ✗ '{sub}' -> MISSING FROM PALETTE")
            else:
                print(f"  (empty/NaN)")

print("\n=== Categories in the filtered data ===")
for i in range(1, TOP_K + 1):
    col = f"category_{i}"
    if col in topk_df_f.columns:
        unique_cats = topk_df_f[col].unique()
        print(f"\n{col}:")
        for cat in unique_cats:
            if cat and str(cat).strip():
                color = palette_cat.get(cat, None)
                if color:
                    print(f"  ✓ '{cat}' -> {color}")
                else:
                    print(f"  ✗ '{cat}' -> MISSING FROM PALETTE")
            else:
                print(f"  (empty/NaN)")


=== Subcategories in the filtered data ===

subcategory_1:
  ✓ 'Neurodegenerative disease' -> (0.4196078431372549, 0.43137254901960786, 0.8117647058823529)
  ✓ 'Replication and repair' -> (0.7803921568627451, 0.7803921568627451, 0.7803921568627451)
  ✓ 'Signal transduction' -> (0.7725490196078432, 0.6901960784313725, 0.8352941176470589)
  ✓ 'Lipid metabolism' -> (0.8901960784313725, 0.4666666666666667, 0.7607843137254902)
  ✓ 'Infectious disease: viral' -> (0.17254901960784313, 0.6274509803921569, 0.17254901960784313)
  ✓ 'Development and regeneration' -> (0.8588235294117647, 0.8588235294117647, 0.5529411764705883)
  ✓ 'Endocrine system' -> (0.6196078431372549, 0.8549019607843137, 0.8980392156862745)

subcategory_2:
  ✓ 'Cardiovascular disease' -> (0.596078431372549, 0.8745098039215686, 0.5411764705882353)
  ✓ 'Cancer: specific types' -> (0.7686274509803922, 0.611764705882353, 0.5803921568627451)
  ✓ 'Transport and catabolism' -> (1.0, 0.4980392156862745, 0.054901960784313725)
  ✓ 'Ami

In [9]:
# Check what subcategories would be in the legend
cat_cols_check = [f"category_{i+1}" for i in range(TOP_K)]
sub_cols_check = [f"subcategory_{i+1}" for i in range(TOP_K)]

all_cat_labels = set(topk_df_f[cat_cols_check].astype(str).values.ravel().tolist())
all_sub_labels = set(topk_df_f[sub_cols_check].astype(str).values.ravel().tolist())

used_cats = sorted([x for x in all_cat_labels if x and x in palette_cat])
used_subs = sorted([x for x in all_sub_labels if x and x in palette_sub])

print("Categories that will be in legend:")
for cat in used_cats:
    print(f"  - {cat}: {palette_cat[cat]}")

print("\nSubcategories that will be in legend:")
for sub in used_subs:
    print(f"  - {sub}: {palette_sub[sub]}")


Categories that will be in legend:
  - Cellular Processes: (1.0, 0.4980392156862745, 0.054901960784313725)
  - Environmental Information Processing: (0.17254901960784313, 0.6274509803921569, 0.17254901960784313)
  - Genetic Information Processing: (0.596078431372549, 0.8745098039215686, 0.5411764705882353)
  - Human Diseases: (0.12156862745098039, 0.4666666666666667, 0.7058823529411765)
  - Metabolism: (1.0, 0.7333333333333333, 0.47058823529411764)
  - Organismal Systems: (0.6823529411764706, 0.7803921568627451, 0.9098039215686274)

Subcategories that will be in legend:
  - Amino acid metabolism: (0.8392156862745098, 0.3803921568627451, 0.4196078431372549)
  - Cancer: specific types: (0.7686274509803922, 0.611764705882353, 0.5803921568627451)
  - Cardiovascular disease: (0.596078431372549, 0.8745098039215686, 0.5411764705882353)
  - Development and regeneration: (0.8588235294117647, 0.8588235294117647, 0.5529411764705883)
  - Digestive system: (0.807843137254902, 0.42745098039215684, 0

In [8]:
# Check how many UNIQUE modules are currently selected
print(f"TOP_MODULES setting: {TOP_MODULES}")
print(f"Number of unique modules selected: {len(top_n_modules)}")
print(f"Unique module names: {top_n_modules}")
print(f"\nTotal rows in filtered data (modules × tissues): {len(corr_wide_f)}")
if "tissue" in corr_wide_f.columns:
    print(f"Breakdown by tissue:")
    print(corr_wide_f.groupby(mcol_corr)["tissue"].apply(list).to_dict())

TOP_MODULES setting: 10
Number of unique modules selected: 10
Unique module names: ['M50', 'M92', 'M92', 'M245', 'M362', 'M245', 'M52', 'M15', 'M122', 'M233']

Total rows in filtered data (modules × tissues): 31
Breakdown by tissue:
{'M50': ['MFBA9BA46', 'PCGBA23', 'CROSS', 'AC'], 'M92': ['PCGBA23', 'CROSS', 'MFBA9BA46'], 'M245': ['PCGBA23', 'CROSS', 'AC', 'MFBA9BA46'], 'M362': ['PCGBA23', 'AC', 'MFBA9BA46', 'CROSS'], 'M52': ['MFBA9BA46', 'CROSS', 'PCGBA23', 'AC'], 'M15': ['CROSS', 'MFBA9BA46', 'PCGBA23', 'AC'], 'M122': ['AC', 'CROSS', 'PCGBA23', 'MFBA9BA46'], 'M233': ['MFBA9BA46', 'PCGBA23', 'CROSS', 'AC']}


/var/folders/5g/7gzv8rg14tv7prkqk0v2f3280000gn/T/ipykernel_68117/601260357.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(corr_wide_f.groupby(mcol_corr)["tissue"].apply(list).to_dict())
